# DBSCAN — Density-Based Spatial Clustering of Applications with Noise

DBSCAN (Ester et al., 1996) is a density-based clustering algorithm that can find arbitrarily-shaped clusters and explicitly identifies noise points. Unlike K-Means, it requires no specification of the number of clusters.

---

## Table of Contents
1. [Motivation — Why Density-Based?](#1-motivation)
2. [Core Concepts and Definitions](#2-definitions)
3. [The DBSCAN Algorithm](#3-algorithm)
4. [Mathematical Formalization](#4-math)
5. [Choosing ε and min_samples](#5-hyperparameters)
6. [Time and Space Complexity](#6-complexity)
7. [DBSCAN vs K-Means vs Hierarchical](#7-comparison)
8. [Limitations and Extensions](#8-limitations)
9. [Implementation from Scratch](#9-scratch)
10. [sklearn DBSCAN on Real Data](#10-sklearn)
11. [Summary](#11-summary)

---
## 1. Motivation — Why Density-Based?

### Failure Cases of Previous Methods

Both K-Means and hierarchical clustering (with Ward linkage) rely on **compactness and Euclidean distance**. They fail when:

1. **Clusters have arbitrary shapes** (spirals, crescents, rings)
2. **Clusters have varying densities**
3. **There are noise points and outliers** that don't belong to any cluster

### The Density Intuition

> A cluster is a dense region of points, separated from other dense regions by sparse regions.

This matches human intuition: if you look at a map of city lights at night, you see dense regions (cities) separated by sparse regions (countryside). DBSCAN formalizes exactly this.

### What Makes DBSCAN Special
1. **No K needed** — number of clusters is discovered automatically
2. **Arbitrary cluster shapes** — follows the density, not distance to a centroid
3. **Outlier detection** — noise points are explicitly labeled as such
4. **Deterministic** — same result every run (no random initialization)

---
## 2. Core Concepts and Definitions

DBSCAN has two hyperparameters:
- **ε (eps)**: the radius of the neighborhood around each point
- **min_samples (MinPts)**: minimum number of points (including the point itself) needed within radius ε to be considered a **core point**

### Definition 1 — ε-Neighborhood

$$N_\varepsilon(p) = \{ q \in D \mid d(p, q) \leq \varepsilon \}$$

The set of all points within distance $\varepsilon$ of point $p$.

### Definition 2 — Core Point

$$p \text{ is a core point} \iff |N_\varepsilon(p)| \geq \text{min\_samples}$$

A core point has a dense neighborhood. It sits **inside** a cluster.

### Definition 3 — Border Point

A point that is:
- **Not** a core point (fewer than min_samples neighbors in its ε-ball)
- **Within** the ε-neighborhood of at least one core point

Border points are on the **edge** of a cluster.

### Definition 4 — Noise Point

A point that is neither a core point nor a border point. It is not within ε of any core point. Labeled as **-1** in sklearn.

### Definition 5 — Directly Density-Reachable

Point $q$ is **directly density-reachable** from point $p$ (with respect to ε, min_samples) if:
1. $q \in N_\varepsilon(p)$ (q is in p's ε-neighborhood)
2. $p$ is a core point

Note: this relation is **not symmetric** — q might be reachable from p but not vice versa (if q is not a core point).

### Definition 6 — Density-Reachable

Point $q$ is **density-reachable** from $p$ if there is a chain of points $p_1, p_2, \dots, p_n$ where:
- $p_1 = p$, $p_n = q$
- Each $p_{i+1}$ is directly density-reachable from $p_i$

### Definition 7 — Density-Connected

Points $p$ and $q$ are **density-connected** if there exists a point $o$ such that both $p$ and $q$ are density-reachable from $o$.

Density-connection **is symmetric** — it's the basis for cluster membership.

### Definition 8 — Cluster

A cluster $C$ is a maximal set of density-connected points:
1. **Connectivity**: All points in $C$ are density-connected to each other
2. **Maximality**: If $p \in C$ and $q$ is density-reachable from $p$, then $q \in C$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle

# Small example to visualize core/border/noise
np.random.seed(0)
X_viz = np.array([
    [1, 1], [1.3, 1.1], [0.9, 1.4], [1.1, 0.8], [1.2, 1.3],  # dense cluster
    [3, 3], [3.2, 2.8], [2.9, 3.1],  # another cluster
    [5, 1],   # noise
    [2, 1.5]  # border
])

eps = 0.6
min_samples = 4

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

db = DBSCAN(eps=eps, min_samples=min_samples).fit(X_viz)
core_mask = np.zeros(len(X_viz), dtype=bool)
core_mask[db.core_sample_indices_] = True

fig, ax = plt.subplots(1, 1, figsize=(9, 7))

for i, (pt, is_core, label) in enumerate(zip(X_viz, core_mask, db.labels_)):
    if label == -1:
        color, marker, size, zorder = 'black', 'x', 120, 3
    elif is_core:
        color, marker, size, zorder = ['#e74c3c','#3498db'][label], 'o', 100, 3
        circ = Circle(pt, eps, fill=False, linestyle='--',
                      color=color, alpha=0.3, linewidth=1)
        ax.add_patch(circ)
    else:
        color, marker, size, zorder = ['#e74c3c','#3498db'][label], 's', 80, 3

    ax.scatter(*pt, c=color, marker=marker, s=size, zorder=zorder)
    ax.annotate(f' p{i}', pt, fontsize=9)

legend_elements = [
    mpatches.Patch(color='#e74c3c', label='Cluster 0 core'),
    mpatches.Patch(color='#3498db', label='Cluster 1 core'),
    plt.Line2D([0],[0], marker='s', color='w', markerfacecolor='#e74c3c', label='Border point'),
    plt.Line2D([0],[0], marker='x', color='black', label='Noise point'),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_title(f'DBSCAN Point Types (ε={eps}, min_samples={min_samples})', fontsize=13)
ax.set_xlim(-0.5, 6.5); ax.set_ylim(0, 4)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('dbscan_point_types.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. The DBSCAN Algorithm

```
Input: D (dataset), ε (radius), MinPts (min neighbors)
Output: cluster labels for each point

label = 0  (current cluster ID)
Mark all points as UNVISITED

FOR each unvisited point P in D:
    Mark P as VISITED
    neighbors = RangeQuery(D, P, ε)  # all points within ε of P
    
    IF |neighbors| < MinPts:
        Mark P as NOISE  # tentative — may be reclassified as border
    ELSE:
        label = label + 1  # new cluster
        ExpandCluster(D, P, neighbors, label, ε, MinPts)

ExpandCluster(D, P, neighbors, C, ε, MinPts):
    Add P to cluster C
    seed_set = neighbors  # points to process
    
    FOR each point Q in seed_set:
        IF Q is NOISE:
            Assign Q to cluster C  # reclassified as border point
        IF Q is UNVISITED:
            Mark Q as VISITED
            Q_neighbors = RangeQuery(D, Q, ε)
            IF |Q_neighbors| >= MinPts:
                seed_set = seed_set ∪ Q_neighbors  # expand the frontier
            Assign Q to cluster C
```

### Key Insight: BFS on the Density Graph

DBSCAN is essentially a **Breadth-First Search** (BFS) where two points are "connected" if they are density-reachable. Each connected component in this density-reachability graph becomes one cluster.

---
## 4. Mathematical Formalization

### Theorem 1 (Correctness)

Let $p$ be a core point and $C$ the cluster discovered by DBSCAN starting from $p$. Then $C$ equals the set of all points density-connected to $p$.

### Theorem 2 (Uniqueness)

Given fixed ε and min_samples, the clustering result is **unique** (up to border point assignment when a border point is within ε of multiple clusters — in the original paper, border points may be assigned to different clusters depending on visit order).

### Border Point Ambiguity

If a border point $b$ is within ε of core points from two different clusters, it could be assigned to either. This is a rare case and doesn't affect core cluster structure. sklearn resolves this deterministically based on the order points are visited.

### DBSCAN as a Graph Problem

Construct a graph $G = (V, E)$ where:
- $V$ = all data points
- $E$ = edge $(p, q)$ if $d(p,q) \leq \varepsilon$ and at least one of $p, q$ is a core point

The connected components of $G$ restricted to core points define the clusters. Border points are added to the cluster of their neighboring core point.

---
## 5. Choosing ε and min_samples

### Rule of Thumb for min_samples

$$\text{min\_samples} \geq d + 1$$

where $d$ = number of features. A common choice is `min_samples = 2 * d`. For 2D data, min_samples = 4 is typical. Higher min_samples → more noise points (more conservative).

### The k-Distance Plot for ε

For each point, compute the distance to its $k$-th nearest neighbor where $k = \text{min\_samples} - 1$. Sort these distances and plot them.

- In the **dense region**: $k$-distances are small (points have close neighbors)
- In the **sparse region**: $k$-distances jump sharply (outliers or cluster boundaries)

**The elbow of the k-distance plot** is the right value for ε:
- Values of ε below the elbow → too strict, most points become noise
- Values of ε above the elbow → too loose, everything merges into one cluster

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_moons = StandardScaler().fit_transform(X_moons)

min_samples = 5
k = min_samples - 1

nbrs = NearestNeighbors(n_neighbors=k)
nbrs.fit(X_moons)
distances, _ = nbrs.kneighbors(X_moons)
k_distances = np.sort(distances[:, k-1])  # distance to k-th nearest neighbor

# Find elbow via max second derivative
d2 = np.diff(np.diff(k_distances))
elbow_idx = np.argmax(d2) + 1
eps_suggested = k_distances[elbow_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(k_distances, color='steelblue', lw=1.5)
axes[0].axhline(eps_suggested, color='red', linestyle='--',
                label=f'Suggested ε = {eps_suggested:.3f}')
axes[0].axvline(elbow_idx, color='red', linestyle=':', alpha=0.5)
axes[0].set_xlabel('Points sorted by distance')
axes[0].set_ylabel(f'{k}-NN distance')
axes[0].set_title(f'k-Distance Plot (k={k}, min_samples={min_samples})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Apply DBSCAN with found eps
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=eps_suggested, min_samples=min_samples)
labels = db.fit_predict(X_moons)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='Set1', s=20, alpha=0.8)
axes[1].set_title(f'DBSCAN Result (ε={eps_suggested:.2f}, min_samples={min_samples})\n'
                  f'{n_clusters} clusters, {n_noise} noise points')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dbscan_eps_selection.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Time and Space Complexity

### Naive Implementation

- Each `RangeQuery` (find all points within ε) takes $O(n)$ by brute force
- Total: $O(n^2)$ — feasible for $n \lesssim 10{,}000$

### With Spatial Index (k-d tree or ball tree)

- `RangeQuery` takes $O(\log n)$ on average for low-dimensional data
- Total: $O(n \log n)$ — feasible for millions of points
- sklearn uses ball trees or k-d trees via `algorithm='ball_tree'` or `'kd_tree'`

### Space Complexity

- $O(n)$ — only need to store point labels, not a distance matrix
- Much more memory-efficient than hierarchical clustering's $O(n^2)$

### High-Dimensional Data

Spatial indices degrade in high dimensions (curse of dimensionality). For $d > 20$, brute force is often faster than a tree. Apply PCA first to reduce dimensionality before DBSCAN.

---
## 7. DBSCAN vs K-Means vs Hierarchical

| Aspect | K-Means | Hierarchical | DBSCAN |
|---|---|---|---|
| K required? | Yes | No (post-hoc) | No |
| Cluster shape | Spherical | Flexible | **Any shape** |
| Outlier handling | None | Partial | **Explicit noise label** |
| Scale sensitivity | Yes — scale first | Yes | Yes — scale first |
| Large datasets | Yes — $O(nKdT)$ | No — $O(n^2)$ | Yes — $O(n \log n)$ |
| Deterministic? | No | Yes | Yes |
| Variable density | No | Partial | **Struggles** (HDBSCAN is better) |
| Parameters | K | linkage | ε, min_samples |

---
## 8. Limitations and Extensions

### 8.1 Varying Density Problem

DBSCAN uses a single global ε. If clusters have very different densities, a small ε splits dense clusters and misses sparse ones; a large ε merges sparse clusters with noise.

**HDBSCAN** (Hierarchical DBSCAN) solves this by:
1. Running DBSCAN across all ε values simultaneously
2. Building a cluster hierarchy
3. Extracting the most stable clusters across the hierarchy

### 8.2 High Dimensionality

ε-neighborhoods become meaningless in high dimensions (all points are roughly equidistant). Reduce dimensionality with PCA first.

### 8.3 Sensitivity to ε

Small changes in ε can drastically change results. The k-distance plot helps, but requires judgment.

In [ ]:
# Demonstrating DBSCAN sensitivity to epsilon
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

X_demo, _ = make_moons(300, noise=0.07, random_state=42)
X_demo = StandardScaler().fit_transform(X_demo)

eps_values = [0.1, 0.3, 0.5, 0.8]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, eps in zip(axes, eps_values):
    lbl = DBSCAN(eps=eps, min_samples=5).fit_predict(X_demo)
    n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_n = (lbl == -1).sum()
    ax.scatter(X_demo[:, 0], X_demo[:, 1], c=lbl, cmap='Set1', s=15, alpha=0.8)
    ax.set_title(f'ε={eps}\n{n_c} clusters, {n_n} noise', fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('DBSCAN Sensitivity to ε (min_samples=5)', fontsize=13)
plt.tight_layout()
plt.savefig('dbscan_epsilon_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. Implementation from Scratch

In [ ]:
import numpy as np
from collections import deque

class DBSCANScratch:
    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples

    def _range_query(self, X, p_idx):
        diffs = X - X[p_idx]
        dists = np.sqrt((diffs**2).sum(axis=1))
        return np.where(dists <= self.eps)[0].tolist()

    def fit_predict(self, X):
        n = len(X)
        labels = np.full(n, -2, dtype=int)  # -2 = unvisited
        cluster_id = 0

        for i in range(n):
            if labels[i] != -2:
                continue
            neighbors = self._range_query(X, i)
            if len(neighbors) < self.min_samples:
                labels[i] = -1  # tentative noise
                continue

            # Expand cluster using BFS
            labels[i] = cluster_id
            seeds = deque(neighbors)
            while seeds:
                j = seeds.popleft()
                if labels[j] == -1:
                    labels[j] = cluster_id  # reclassify noise as border
                if labels[j] != -2:
                    continue
                labels[j] = cluster_id
                j_neighbors = self._range_query(X, j)
                if len(j_neighbors) >= self.min_samples:
                    seeds.extend(j_neighbors)

            cluster_id += 1

        self.labels_ = labels
        return labels


# Compare with sklearn
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

X_t, _ = make_moons(150, noise=0.07, random_state=42)
X_t = StandardScaler().fit_transform(X_t)

eps, min_s = 0.35, 5
scratch = DBSCANScratch(eps=eps, min_samples=min_s)
labels_s = scratch.fit_predict(X_t)

labels_sk = DBSCAN(eps=eps, min_samples=min_s).fit_predict(X_t)

ari = adjusted_rand_score(labels_sk, labels_s)
print(f"ARI vs sklearn: {ari:.4f} (1.0 = identical)")
print(f"Scratch  — clusters: {len(set(labels_s))-(1 if -1 in labels_s else 0)}, noise: {(labels_s==-1).sum()}")
print(f"sklearn  — clusters: {len(set(labels_sk))-(1 if -1 in labels_sk else 0)}, noise: {(labels_sk==-1).sum()}")

---
## 10. sklearn DBSCAN on Real Data

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

datasets = [
    ('Blobs', *make_blobs(300, centers=3, cluster_std=0.6, random_state=42), 0.35, 5),
    ('Moons', *make_moons(300, noise=0.05, random_state=42), 0.25, 5),
    ('Circles', *make_circles(300, noise=0.04, factor=0.5, random_state=42), 0.2, 5),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, X_d, y_d, eps, ms) in zip(axes, datasets):
    X_s = StandardScaler().fit_transform(X_d)
    labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(X_s)
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    n_n = (labels == -1).sum()

    # noise points in black
    noise_mask = labels == -1
    ax.scatter(X_s[noise_mask, 0], X_s[noise_mask, 1], c='black', s=20, marker='x', label='Noise')
    ax.scatter(X_s[~noise_mask, 0], X_s[~noise_mask, 1],
               c=labels[~noise_mask], cmap='Set1', s=25, alpha=0.9)
    ax.set_title(f'{name}\nε={eps}, min_s={ms} → {n_c} clusters, {n_n} noise', fontsize=11)
    ax.grid(True, alpha=0.3)
    if n_n > 0:
        ax.legend(fontsize=9)

plt.suptitle('DBSCAN Handles Arbitrary Cluster Shapes', fontsize=13)
plt.tight_layout()
plt.savefig('dbscan_shapes.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 11. Summary

| Concept | Key Point |
|---|---|
| Core point | Has ≥ min_samples neighbors within radius ε |
| Border point | Within ε of a core point, but not a core point itself |
| Noise point | Not within ε of any core point — labeled -1 |
| Cluster | Maximal set of density-connected points |
| ε selection | k-distance plot elbow (k = min_samples - 1) |
| min_samples | Rule of thumb: 2 × n_features |
| Cluster shape | Any shape — follows density, not distance to centroid |
| Complexity | $O(n \log n)$ with spatial index, $O(n^2)$ brute force |
| Limitation | Single global ε fails for varying-density clusters → use HDBSCAN |

### sklearn Quick Reference
```python
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
db = DBSCAN(eps=0.5, min_samples=5, algorithm='ball_tree')
labels = db.fit_predict(X_scaled)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_points = X_scaled[labels == -1]   # outliers
core_indices = db.core_sample_indices_  # indices of core points
```